# Restoration Fusion: Colab backend
Runs three frozen pretrained restorers (HINet, SFNet, AdaIR) plus our fusion module
behind a small FastAPI server, and exposes it through a public tunnel.
The React frontend only sends one image and shows the four outputs.

**Run the cells top to bottom on a GPU runtime (T4 is enough).**
Cells marked CONFIG are the ones you may need to edit.

## 1. Setup

In [6]:
import os, sys, re, json, glob, io, time, base64, shutil, subprocess, threading
import torch
assert torch.cuda.is_available(), "Switch to a GPU runtime: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "einops", "fastapi", "uvicorn",
                "python-multipart", "scikit-image", "lmdb", "pyyaml", "gdown", "huggingface_hub",
                "gguf", "pytorch_msssim", "opencv-python-headless"], check=True)

GPU: Tesla T4


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', '-q', 'install', 'einops', 'fastapi', 'uvicorn', 'python-multipart', 'scikit-image', 'lmdb', 'pyyaml', 'gdown', 'huggingface_hub', 'gguf', 'pytorch_msssim', 'opencv-python-headless'], returncode=0)

## 2. CONFIG: weights and repos
* HINet weights are fetched automatically from the official Drive files.
* **AdaIR and SFNet weights now both come from Hugging Face** (no Drive quota). AdaIR uses a public repo;
  SFNet uses *your own* repo set in `SFNET_HF_REPO`. If you have not made it yet, run the optional
  **one-time cell** right below the CONFIG cell. After that, every run downloads SFNet automatically.
* Fusion checkpoints (`fusion_deblur.pth`) is the trained **DFFN** weights from the DFFN notebooks.
  Either upload them to `/content/weights/`, or set `FUSION_FROM_DRIVE = True` in the CONFIG cell to copy them from your Drive `DFFN` folder.
  Without them the fusion module is an untrained placeholder that behaves like a plain average of the
  available models, and the API reports `fusion_trained: false`.

In [7]:
WORK = "/content"
print("NOTEBOOK VERSION: sfnet-hf-v8-dffn-deblur-only (2026-11-26)")   # if you do not see this line, you are running an old copy
WEIGHTS = f"{WORK}/weights"; os.makedirs(WEIGHTS, exist_ok=True)

# --- SFNet checkpoint ---
# Set a path to override everything (otherwise it is fetched automatically):
SFNET_DEBLUR_CKPT = ""     # e.g. "/content/weights/sfnet/GOPRO.pkl"

# Your own Hugging Face model repo holding GOPRO.pkl (same mechanism as AdaIR, no quota).
# Create it once with the "ONE-TIME" cell below, then paste its id here, e.g. "yourname/sfnet-weights".
# A public repo needs no token. For a private repo add a Colab secret named HF_TOKEN.
SFNET_HF_REPO = ""
SFNET_FILES = {"deblur": "GOPRO.pkl"}   # exact official file name

# Optional fallbacks, tried only if the HF repo is not set / not reachable:
SFNET_MY_DRIVE_DIR = ""    # e.g. "/MyDrive/sfnet_weights" (a folder in YOUR Drive holding GOPRO.pkl)
SFNET_OFFICIAL_IDS = {"GOPRO.pkl": "1KjetPk7dmSod8ntzlo3OU6D5oPPqjMSe"}     # authors' file: usually quota-blocked

# --- AdaIR: direct PyTorch checkpoint URL (preferred) or the HF GGUF conversion used in your notebook ---
ADAIR_CKPT_URL = ""
ADAIR_GGUF_REPO = "cstr/adair-GGUF"

# --- DFFN fusion checkpoint (state_dict .pth/.pt from the DFFN training notebook). ---
FUSION_CKPT = {"deblur": f"{WEIGHTS}/fusion_deblur.pth"}
# Optional: copy it from your Drive (mounts Drive, one pop-up). Looks for
#   <DFFN_DRIVE_ROOT>/results/final/ckpt/{task}_DFFN.pt   then   <DFFN_DRIVE_ROOT>/checkpoints/final/{task}_dffn.pt
FUSION_FROM_DRIVE = True   # final DFFN weights live in your Drive (one pop-up)
DFFN_DRIVE_ROOT = "/content/drive/MyDrive/DFFN"

# --- Serving ---
MAX_SIDE = 1024          # longest image side processed (bigger inputs are downscaled to fit a T4)
PORT = 8000

NOTEBOOK VERSION: sfnet-hf-v8-dffn-deblur-only (2026-11-26)


### ONE-TIME: publish SFNet weights to your Hugging Face repo (skip once `SFNET_HF_REPO` is set)
Google Drive blocks the authors' SFNet files, but Hugging Face does not (that is why AdaIR works).
1. In your browser, download `GOPRO.pkl` from the links in the SFNet README (the browser works even when `gdown` is blocked).
2. Create a free token with **write** access at https://huggingface.co/settings/tokens and add it as a Colab secret named `HF_TOKEN` (key icon in the left sidebar, enable notebook access).
3. Run the next cell, choose the two files when asked, and copy the printed repo id into `SFNET_HF_REPO` in the CONFIG cell.

In [9]:

# ONE-TIME: Upload GOPRO.pkl from Google Drive to Hugging Face
# Automatically mounts Drive and uploads the file.
# No repeated manual upload required.

import os
from pathlib import Path
from google.colab import drive, userdata
from huggingface_hub import HfApi

# --------------------------------------------------
# 1. Mount Google Drive
# --------------------------------------------------

drive.mount("/content/drive", force_remount=True)

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    raise RuntimeError("Google Drive is not mounted properly.")

print("Google Drive connected!")

# --------------------------------------------------
# 2. Hugging Face configuration
# --------------------------------------------------

HF_REPO_NAME = "sfnet-weights"

tok = userdata.get("HF_TOKEN")

if not tok:
    raise ValueError(
        "HF_TOKEN is missing. Add it in Colab Secrets "
        "with WRITE access."
    )

api = HfApi(token=tok)

user = api.whoami()["name"]
repo_id = f"{user}/{HF_REPO_NAME}"

# --------------------------------------------------
# 3. Create Hugging Face repository
# --------------------------------------------------

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=False,
    exist_ok=True
)

print("Hugging Face repository:", repo_id)

# --------------------------------------------------
# 4. Find GOPRO.pkl in Google Drive
# --------------------------------------------------

matches = list(DRIVE_ROOT.rglob("GOPRO.pkl"))

if not matches:
    raise FileNotFoundError(
        "GOPRO.pkl not found in Google Drive.\n"
        "Place it inside MyDrive/DFFN/weights/"
    )

file_path = matches[0]

print("Found checkpoint:", file_path)
print("File size:", round(file_path.stat().st_size / 1e6, 2), "MB")

# --------------------------------------------------
# 5. Upload to Hugging Face
# --------------------------------------------------

api.upload_file(
    path_or_fileobj=str(file_path),
    path_in_repo="GOPRO.pkl",
    repo_id=repo_id,
    repo_type="model"
)

print("\nGOPRO.pkl uploaded successfully!")
print("Repository:", f"https://huggingface.co/{repo_id}")

# --------------------------------------------------
# 6. Configuration value for your project
# --------------------------------------------------

print(f'\nSet SFNET_HF_REPO = "{repo_id}" in your CONFIG cell.')


Mounted at /content/drive
Google Drive connected!
Hugging Face repository: Shreyaaa1735/sfnet-weights
Found checkpoint: /content/drive/MyDrive/DFFN/weights/GOPRO.pkl
File size: 160.23 MB


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ve/DFFN/weights/GOPRO.pkl:   0%|          |  166kB /  160MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.



GOPRO.pkl uploaded successfully!
Repository: https://huggingface.co/Shreyaaa1735/sfnet-weights

Set SFNET_HF_REPO = "Shreyaaa1735/sfnet-weights" in your CONFIG cell.


In [10]:
def sh(cmd):
    print("$", cmd); return subprocess.run(cmd, shell=True, check=False)

for name, url in {"HINet": "https://github.com/megvii-model/HINet.git",
                  "SFNet": "https://github.com/c-yn/SFNet.git",
                  "AdaIR": "https://github.com/c-yn/AdaIR.git"}.items():
    if not os.path.isdir(f"{WORK}/{name}"):
        sh(f"git clone --depth 1 {url} {WORK}/{name}")

# basicsr/version.py is normally generated by setup.py (same workaround as your HINet notebook)
vp = f"{WORK}/HINet/basicsr/version.py"
if not os.path.exists(vp):
    open(vp, "w").write("__version__ = '1.2.0'\nshort_version = '1.2.0'\nversion_info = (1, 2, 0)\n")

# HINet official checkpoint (GoPro deblurring)
HINET_CKPT = {"deblur": f"{WEIGHTS}/HINet-GoPro.pth"}
for path, gid in [(HINET_CKPT["deblur"], "1dw8PKVkLfISzNtUu3gqGh83NBO83ZQ5n")]:
    if not (os.path.exists(path) and os.path.getsize(path) > 1_000_000):
        sh(f'gdown "https://drive.google.com/uc?id={gid}" -O {path}')

# ---------------- SFNet weights ----------------
# Order: manual path > your HF repo (same as AdaIR) > your Drive copy > authors' Drive file (quota-prone).
# Only the exact official file GOPRO.pkl is ever used. Other files in the authors' folder
# (csd.pkl, rsblur.pkl, SRRS.pkl, ...) are weights for OTHER tasks and are never picked as a fallback.
def _ok(p): return bool(p) and os.path.isfile(p) and os.path.getsize(p) > 1_000_000

def _hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

def sfnet_fetch(fname, dest):
    os.makedirs(dest, exist_ok=True)
    out = os.path.join(dest, fname)
    if _ok(out): return out
    # 1) your Hugging Face repo (exactly how AdaIR is fetched)
    if SFNET_HF_REPO:
        try:
            from huggingface_hub import hf_hub_download
            p = hf_hub_download(SFNET_HF_REPO, fname, local_dir=dest, token=_hf_token())
            if _ok(p): print(f"SFNet: {fname} from Hugging Face ({SFNET_HF_REPO})"); return p
        except Exception as ex: print(f"SFNet: HF fetch failed for {fname}: {type(ex).__name__}: {str(ex)[:200]}")
    # 2) a copy in your own Drive
    if SFNET_MY_DRIVE_DIR:
        try:
            from google.colab import drive
            if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
            src = "/content/drive" + SFNET_MY_DRIVE_DIR + "/" + fname
            if os.path.exists(src): shutil.copy(src, out); print("SFNet: copied from your Drive:", src); return out
            print("SFNet: not found in your Drive:", src)
        except Exception as ex: print("SFNet: Drive copy failed:", type(ex).__name__, ex)
    # 3) authors' Drive file, exact id only (no folder download)
    gid = SFNET_OFFICIAL_IDS.get(fname)
    if gid:
        sh(f'gdown "https://drive.google.com/uc?id={gid}" -O {out}')
        if _ok(out): return out
        if os.path.exists(out): os.remove(out)      # drop a tiny HTML error page if gdown saved one
    return ""

if not SFNET_DEBLUR_CKPT: SFNET_DEBLUR_CKPT = sfnet_fetch(SFNET_FILES["deblur"], f"{WEIGHTS}/sfnet")
print("SFNet deblur ckpt:", SFNET_DEBLUR_CKPT or "NOT FOUND (SFNet will be skipped)")
if not SFNET_DEBLUR_CKPT:
    print("\n>>> SFNet weights missing. The authors' Drive file hits Google's download quota. One-time fix:\n"
          "    1. Download the file in your browser (signed in to Google):\n"
          "       GOPRO.pkl      https://drive.google.com/uc?id=1KjetPk7dmSod8ntzlo3OU6D5oPPqjMSe\n"
          "    2. Run the 'ONE-TIME: publish the SFNet weights to your Hugging Face repo' cell above,\n"
          "       paste the printed repo id into SFNET_HF_REPO, and re-run the CONFIG and this cell.")

# AdaIR weights
ADAIR_CKPT = ""
if ADAIR_CKPT_URL:
    ADAIR_CKPT = f"{WEIGHTS}/adair5d.ckpt"
    if not os.path.exists(ADAIR_CKPT):
        sh(f'wget -q "{ADAIR_CKPT_URL}" -O {ADAIR_CKPT}')
else:
    from huggingface_hub import hf_hub_download, list_repo_files
    ggufs = [f for f in list_repo_files(ADAIR_GGUF_REPO) if f.lower().endswith(".gguf")]
    chosen = ([f for f in ggufs if "f32" in f.lower()] or ggufs)[0]
    ADAIR_CKPT = hf_hub_download(ADAIR_GGUF_REPO, chosen, local_dir=WEIGHTS)
print("AdaIR ckpt:", ADAIR_CKPT)

$ git clone --depth 1 https://github.com/megvii-model/HINet.git /content/HINet
$ git clone --depth 1 https://github.com/c-yn/SFNet.git /content/SFNet
$ git clone --depth 1 https://github.com/c-yn/AdaIR.git /content/AdaIR
$ gdown "https://drive.google.com/uc?id=1dw8PKVkLfISzNtUu3gqGh83NBO83ZQ5n" -O /content/weights/HINet-GoPro.pth
$ gdown "https://drive.google.com/uc?id=1KjetPk7dmSod8ntzlo3OU6D5oPPqjMSe" -O /content/weights/sfnet/GOPRO.pkl
SFNet deblur ckpt: /content/weights/sfnet/GOPRO.pkl


adair-5d-f32.gguf: reconstructing file:   0%|          |  0.00B /  115MB            

adair-5d-f32.gguf: downloading bytes:           |  0.00B            

AdaIR ckpt: /content/weights/adair-5d-f32.gguf


## 3. Model wrappers
Every wrapper returns `(restored_image, feature_map)`.
The feature map is the input of the network's last 3-channel output convolution, i.e. the
decoder feature right before it is turned into pixels. If a network has several such layers, the one
that executes last wins (that is the full-resolution output head).

In [11]:
import numpy as np, yaml
import torch.nn as nn, torch.nn.functional as F

dev = "cuda"
CLASH = {"utils", "data", "models", "net", "options", "train", "eval", "main", "layers", "losses"}

def isolated_import(repo_dir, fn):
    """Import from one repo without leaving its generic top-level names (utils, models, ...)
    in sys.modules, because HINet / SFNet / AdaIR all use such names."""
    before = set(sys.modules)
    sys.path.insert(0, repo_dir)
    try:
        return fn()
    finally:
        sys.path.remove(repo_dir)
        for m in list(sys.modules):
            if m not in before and m.split(".")[0] in CLASH:
                del sys.modules[m]

def strip_module(sd):
    return {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}

class FeatureTap:
    """Records the input of the most recently executed Conv2d(out_channels=3)."""
    def __init__(self, net, layer_name=None):
        self.feat, self.layer = None, None
        mods = dict(net.named_modules())
        targets = {layer_name: mods[layer_name]} if layer_name else \
                  {n: m for n, m in mods.items() if isinstance(m, nn.Conv2d) and m.out_channels == 3}
        assert targets, "no 3-channel output conv found; pass layer_name"
        for n, m in targets.items():
            m.register_forward_pre_hook(lambda mod, inp, n=n: self._store(n, inp[0]))
        self.candidates = list(targets)
    def _store(self, name, x):
        self.feat, self.layer = x.detach(), name

class Restorer:
    def __init__(self, key, name, net, tap_layer=None):
        self.key, self.name = key, name
        self.net = net.to(dev).eval()
        for p in self.net.parameters(): p.requires_grad_(False)
        self.tap = FeatureTap(self.net, tap_layer)
        self.feat_channels = None
    @torch.inference_mode()
    def __call__(self, x):
        self.tap.feat = None
        out = self.net(x)
        if isinstance(out, (list, tuple)): out = out[-1]
        feat = self.tap.feat
        self.feat_channels = feat.shape[1]
        return out.clamp(0, 1), feat

# ---------- HINet (official repo, architecture parameters read from the repo's own test YAML) ----------
def load_hinet(task):
    def _imp():
        from basicsr.models.archs.hinet_arch import HINet
        return HINet
    HINet = isolated_import(f"{WORK}/HINet", _imp)
    yml = {"deblur": "options/test/GoPro/HINet-GoPro.yml"}[task]
    cfg = yaml.safe_load(open(f"{WORK}/HINet/{yml}"))["network_g"]; cfg.pop("type")
    net = HINet(**cfg)
    ck = torch.load(HINET_CKPT[task], map_location="cpu")
    net.load_state_dict(strip_module(ck.get("params", ck)), strict=True)
    return Restorer("hinet", "HINet", net)

# ---------- SFNet (official repo, Motion_deblurring sub-folder) ----------
def load_sfnet(task):
    ckpt = {"deblur": SFNET_DEBLUR_CKPT}[task]
    if not ckpt: raise FileNotFoundError(f"no SFNet checkpoint configured for {task}")
    sub = {"deblur": "Motion_deblurring"}[task]
    def _imp():
        import importlib
        for modname in ("models.SFNet", "models.model", "model", "models"):
            try:
                mod = importlib.import_module(modname)
                if hasattr(mod, "build_net"): return mod.build_net
            except ImportError:
                continue
        raise ImportError(f"could not find build_net in {WORK}/SFNet/{sub}; open the folder and adapt load_sfnet")
    build_net = isolated_import(f"{WORK}/SFNet/{sub}", _imp)
    # Motion_deblurring.build_net needs a mode argument; "train" mode = adaptive pooling: same weights, works for any image size
    net = build_net(["train", "GOPRO"])
    ck = torch.load(ckpt, map_location="cpu")
    net.load_state_dict(strip_module(ck.get("model", ck.get("params", ck.get("state_dict", ck)))), strict=True)
    return Restorer("sfnet", "SFNet", net)

# ---------- AdaIR (all-in-one checkpoint) ----------
_ADAIR = {}
def load_adair(task):
    if "net" not in _ADAIR:
        AdaIR = isolated_import(f"{WORK}/AdaIR", lambda: __import__("net.model", fromlist=["AdaIR"]).AdaIR)
        net = AdaIR(decoder=True)
        target = net.state_dict()
        if ADAIR_CKPT.endswith(".gguf"):                       # same conversion logic as your AdaIR notebook
            from gguf import GGUFReader
            raw = {t.name: torch.from_numpy(np.array(t.data, dtype=np.float32, copy=True)) for t in GGUFReader(ADAIR_CKPT).tensors}
        else:
            ck = torch.load(ADAIR_CKPT, map_location="cpu")
            raw = ck.get("state_dict", ck)
        for p in ("net.", "module.net.", "module."):
            if raw and all(k.startswith(p) for k in raw):
                raw = {k[len(p):]: v for k, v in raw.items()}
        tk = list(target)
        if all(k in raw and raw[k].numel() == target[k].numel() for k in tk):
            mapped = {k: raw[k].reshape(target[k].shape).float() for k in tk}
        else:
            rv = list(raw.values()); assert len(rv) == len(tk), "AdaIR weight mapping failed (tensor count differs)"
            assert all(tuple(r.shape) == tuple(target[k].shape) for r, k in zip(rv, tk)), "AdaIR weight mapping failed (shape mismatch)"
            mapped = {k: r.float() for k, r in zip(tk, rv)}
        net.load_state_dict(mapped, strict=True)
        _ADAIR["net"] = net
    return Restorer("adair", "AdaIR", _ADAIR["net"])

LOADERS = {"hinet": load_hinet, "sfnet": load_sfnet, "adair": load_adair}
ORDER = ["hinet", "sfnet", "adair"]
MODELS = {"deblur": {}}
LOAD_NOTES = {"deblur": {}}
for task in MODELS:
    for key in ORDER:
        try:
            MODELS[task][key] = LOADERS[key](task)
            print(f"[ok]   {task:6s} {key}")
        except Exception as e:
            MODELS[task][key] = None
            LOAD_NOTES[task][key] = f"{type(e).__name__}: {str(e)[:200]}"
            print(f"[SKIP] {task:6s} {key}: {LOAD_NOTES[task][key]}")

[ok]   deblur hinet
[ok]   deblur sfnet
[ok]   deblur adair


## 4. Fusion module: DFFN (Deep Feature Fusion Network)
This is the DFFN from the training notebooks, dropped in unchanged (same layers and parameter names, so the trained checkpoints load with `strict=True`).
* **Align**: 1x1 conv per expert maps its tap feature (HINet 64, SFNet 32, AdaIR 96 channels) to a common width of 48.
* **Gate**: a small conv head predicts one weight per expert per pixel (softmax over experts). These are the trust maps.
* **Fuse + refine**: weighted feature sum, squeeze-and-excitation, then 4 residual blocks.
* **Output**: gate-weighted blend of the experts' restored images plus a residual correction from the refined features.

The feature taps used by the wrappers above are the same layers DFFN was trained on (HINet `last` input, SFNet `feat_extract[5]` input = `Decoder[2]` output, AdaIR `output` input = `refinement` output).
The final gate conv is zero-initialised here, so an *untrained* module equals a plain average. Trained checkpoints overwrite it.
`forward(imgs, feats, mask)` keeps the server's signature; if an expert is unavailable it is masked out of the softmax.

In [12]:
class SE(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c // r, 1), nn.ReLU(inplace=True),
                                nn.Conv2d(c // r, c, 1), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x)

class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(c, c, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(c, c, 3, padding=1))
    def forward(self, x): return x + self.body(x)

DFFN_CHANNELS = (64, 32, 96)   # HINet, SFNet, AdaIR tap widths the trained DFFN expects

class FusionModule(nn.Module):
    """DFFN, identical layer names/shapes to the training notebooks (use_gate=True)."""
    def __init__(self, feat_channels=DFFN_CHANNELS, dim=48, n_blocks=4):
        super().__init__()
        self.k = len(feat_channels)
        self.align = nn.ModuleList([nn.Conv2d(ci, dim, 1) for ci in feat_channels])
        self.gate = nn.Sequential(nn.Conv2d(self.k * dim, dim, 3, padding=1), nn.ReLU(inplace=True),
                                  nn.Conv2d(dim, self.k, 3, padding=1))
        self.se = SE(dim)
        self.refine = nn.Sequential(*[ResBlock(dim) for _ in range(n_blocks)])
        self.head = nn.Conv2d(dim, 3, 3, padding=1)
        # zero head + zero final gate conv => untrained module == plain average (placeholder); trained ckpt overwrites both
        for m in (self.head, self.gate[-1]):
            nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, imgs, feats, mask):
        """imgs: K tensors [B,3,H,W]; feats: K tensors [B,C_k,h,w] (or None); mask: K bools (model available)."""
        H, W = imgs[0].shape[-2:]
        aligned = []
        for i, (f, ok) in enumerate(zip(feats, mask)):
            if ok and f is not None:
                a = self.align[i](f)
                if a.shape[-2:] != (H, W): a = F.interpolate(a, size=(H, W), mode="bilinear", align_corners=False)
            else:
                a = torch.zeros(imgs[0].shape[0], self.align[i].out_channels, H, W, device=imgs[0].device)
            aligned.append(a)
        logits = self.gate(torch.cat(aligned, 1))
        neg = torch.tensor([0.0 if ok else -1e4 for ok in mask], device=logits.device).view(1, -1, 1, 1)
        w = torch.softmax(logits + neg, dim=1)
        fused = sum(w[:, i:i + 1] * aligned[i] for i in range(self.k))
        fused = self.refine(self.se(fused))
        blend = sum(w[:, i:i + 1] * imgs[i] for i in range(self.k) if mask[i])
        return (blend + self.head(fused)).clamp(0, 1), w

FUSION, FUSION_TRAINED = {}, {}

def fetch_dffn_checkpoints():
    """Optional (FUSION_FROM_DRIVE): copy trained DFFN weights from your Drive into FUSION_CKPT paths."""
    if not FUSION_FROM_DRIVE: return
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
    for t, dst in FUSION_CKPT.items():
        for rel in (f"checkpoints/final/{t}_dffn.pt", f"results/final/ckpt/{t}_DFFN.pt"):
            src = f"{DFFN_DRIVE_ROOT}/{rel}"
            if os.path.exists(src):
                if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
                    shutil.copy(src, dst); print(f"DFFN[{t}]: copied {src}")
                else: print(f"DFFN[{t}]: {dst} is already the newest copy of {src}")
                break
        else:
            print(f"DFFN[{t}]: no checkpoint found under {DFFN_DRIVE_ROOT}")

def init_fusion(task):
    # feature widths are discovered with a tiny dry run so the module matches the real taps
    x = torch.rand(1, 3, 64, 64, device=dev); chans = []
    for key, default in zip(ORDER, DFFN_CHANNELS):
        m = MODELS[task][key]
        if m:
            try: m(x); chans.append(m.feat_channels)
            except Exception as e: print("dry run failed for", task, key, e); chans.append(default)
        else: chans.append(default)
    if tuple(chans) != DFFN_CHANNELS:
        print(f"WARNING fusion[{task}]: tap widths {chans} differ from the {list(DFFN_CHANNELS)} DFFN was trained with; trained weights will not load.")
    net = FusionModule(tuple(chans)).to(dev).eval()
    path = FUSION_CKPT[task]
    if os.path.exists(path):
        ck = torch.load(path, map_location="cpu")
        net.load_state_dict(strip_module(ck.get("params", ck.get("model", ck))), strict=True)
        FUSION_TRAINED[task] = True
    else:
        FUSION_TRAINED[task] = False
    FUSION[task] = net
    print(f"fusion[{task}] feature widths {chans}  trained={FUSION_TRAINED[task]}")

fetch_dffn_checkpoints()
for t in MODELS: init_fusion(t)

print("\nFeature tap layers (check these look like the last decoder feature):")
for t in MODELS:
    for k, m in MODELS[t].items():
        if m: print(f"  {t:6s} {k:6s} -> {m.tap.layer}  ({m.feat_channels} ch)  candidates: {len(m.tap.candidates)}")

fusion[deblur] feature widths [64, 32, 96]  trained=False

Feature tap layers (check these look like the last decoder feature):
  deblur hinet  -> last  (64 ch)  candidates: 2
  deblur sfnet  -> feat_extract.5.main.0  (32 ch)  candidates: 3
  deblur adair  -> output  (96 ch)  candidates: 1


## 5. Pipeline and metrics

In [13]:
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

def load_rgb(data: bytes, max_side=MAX_SIDE, size=None):
    im = Image.open(io.BytesIO(data)).convert("RGB")
    if size is not None:
        if im.size != size: im = im.resize(size, Image.LANCZOS)
    else:
        s = max(im.size)
        if s > max_side:
            r = max_side / s; im = im.resize((max(32, round(im.width * r)), max(32, round(im.height * r))), Image.LANCZOS)
    return im

def pad16(x):   # name kept; pads to a multiple of 32 like the DFFN notebook
    h, w = x.shape[-2:]; ph, pw = (-h) % 32, (-w) % 32
    if ph == 0 and pw == 0: return x, (h, w)
    mode = "reflect" if (ph < h and pw < w) else "replicate"
    return F.pad(x, (0, pw, 0, ph), mode=mode), (h, w)

def to_u8(t):  # [1,3,H,W] -> HxWx3 uint8
    return (t.clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)

def score(pred, gt):   # RGB PSNR / SSIM, as used for GoPro deblurring
    pred, gt = pred.astype(np.float64), gt.astype(np.float64)
    return (float(peak_signal_noise_ratio(gt, pred, data_range=255)),
            float(structural_similarity(gt, pred, channel_axis=2, data_range=255)))

def data_url(u8):
    buf = io.BytesIO(); Image.fromarray(u8).save(buf, "PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

TRUST_COLORS = {"hinet": (59, 130, 246), "sfnet": (245, 158, 11), "adair": (217, 70, 239)}   # same as the website legend

def make_trust(wmap, base, h, w):
    """Colour every pixel by the models the fusion module trusts there, over a greyscale copy of the output."""
    cols = torch.tensor([TRUST_COLORS[k] for k in ORDER], device=wmap.device, dtype=torch.float32).view(1, len(ORDER), 3, 1, 1) / 255.0
    col = (wmap[:, :, None] * cols).sum(1)                                   # [B,3,H,W]
    gray = base.mean(1, keepdim=True).expand(-1, 3, -1, -1)
    ov = (0.35 * gray + 0.65 * col)[..., :h, :w]
    share = wmap[..., :h, :w].mean(dim=(0, 2, 3)).tolist()
    return {"image": data_url(to_u8(ov)), "share": {k: round(100 * v, 1) for k, v in zip(ORDER, share)}}

DISPLAY = {"hinet": "HINet", "sfnet": "SFNet", "adair": "AdaIR", "average": "Simple average", "fusion": "Fusion (ours)"}

@torch.inference_mode()
def run_pipeline(image_bytes, task, gt_bytes=None):
    t_all = time.time()
    im = load_rgb(image_bytes)
    x0 = torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)[None].to(dev)
    x, (h, w) = pad16(x0)
    gt_u8 = np.asarray(load_rgb(gt_bytes, size=im.size)) if gt_bytes else None

    outs, feats, items = {}, {}, []
    for key in ORDER:
        m = MODELS[task][key]; item = {"id": key, "name": DISPLAY[key], "available": False, "note": "", "image": None, "psnr": None, "ssim": None, "time_ms": None}
        if m is None:
            item["note"] = "Not loaded: " + LOAD_NOTES[task].get(key, "unknown")
        else:
            try:
                t = time.time(); torch.cuda.synchronize()
                out, feat = m(x); torch.cuda.synchronize()
                outs[key], feats[key] = out, feat
                item.update(available=True, time_ms=round((time.time() - t) * 1000))
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); item["note"] = "Out of GPU memory. Try a smaller image."
            except Exception as e:
                item["note"] = f"{type(e).__name__}: {str(e)[:160]}"
        items.append(item)

    mask = [k in outs for k in ORDER]

    # Baseline for the comparison: plain mean of the restored images of the models that ran.
    avg_item = {"id": "average", "name": DISPLAY["average"], "available": False, "note": "", "image": None, "psnr": None, "ssim": None, "time_ms": None}
    if outs:
        used = [k for k in ORDER if k in outs]
        outs["average"] = torch.stack([outs[k] for k in used]).mean(0)
        avg_item.update(available=True, time_ms=0)
        if len(used) < len(ORDER):
            avg_item["note"] = "Mean of " + ", ".join(DISPLAY[k] for k in used) + " (the others are unavailable)."
    else:
        avg_item["note"] = "No base model produced an output."
    items.append(avg_item)

    fusion_item ={"id": "fusion", "name": DISPLAY["fusion"], "available": False, "note": "", "image": None, "psnr": None, "ssim": None, "time_ms": None}
    if any(mask):
        try:
            t = time.time(); torch.cuda.synchronize()
            # DFFN was trained/evaluated on unpadded images: crop the experts' outputs and features back to (h, w) first
            fused, wmap = FUSION[task]([outs.get(k, x)[..., :h, :w] for k in ORDER],
                                       [feats[k][..., :h, :w] if k in feats else None for k in ORDER], mask)
            torch.cuda.synchronize()
            outs["fusion"] = fused
            trust = make_trust(wmap, fused, h, w)
            fusion_item.update(available=True, time_ms=round((time.time() - t) * 1000))
            if not FUSION_TRAINED[task]:
                fusion_item["note"] = "Placeholder: fusion weights not trained yet, so this equals a plain average."
        except Exception as e:
            fusion_item["note"] = f"{type(e).__name__}: {str(e)[:160]}"
    else:
        fusion_item["note"] = "No base model produced an output."
    items.append(fusion_item)

    for it in items:
        if it["id"] in outs:
            u8 = to_u8(outs[it["id"]][..., :h, :w])
            it["image"] = data_url(u8)
            if gt_u8 is not None: it["psnr"], it["ssim"] = score(u8, gt_u8)
    del outs, feats; torch.cuda.empty_cache()
    return {"task": task, "width": w, "height": h, "y_channel": False, "has_gt": gt_u8 is not None,
            "fusion_trained": FUSION_TRAINED[task], "trust": trust, "elapsed_ms": round((time.time() - t_all) * 1000),
            "input": data_url(np.asarray(im)), "results": items}

## 6. API server
`GET /health`, `GET /benchmarks`, `POST /restore` (multipart: `image`, `task` = deblur, optional `gt`).

In [14]:
from typing import Optional
from fastapi import FastAPI, File, Form, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware

BENCH_FILE = f"{WORK}/benchmarks.json"
def _rows(v):  # (psnr, ssim) per model, from the DFFN notebook's result table
    return {k: {"psnr": p, "ssim": s} for k, (p, s) in zip(("hinet", "sfnet", "adair", "average", "fusion"), v)}
BENCH_TEMPLATE = {   # measured with the DFFN notebook (deblur: RGB)
    "deblur": {"dataset": "GoPro test", "n_images": 1111,
               "rows": _rows([(32.88, 0.9380), (33.22, 0.9429), (28.11, 0.8607), (32.25, 0.9324), (33.45, 0.9448)])},
}
if not os.path.exists(BENCH_FILE):
    json.dump(BENCH_TEMPLATE, open(BENCH_FILE, "w"), indent=2)   # fill with your real test-set numbers

app = FastAPI(title="Restoration Fusion API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
gpu_lock = threading.Lock()

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0), "max_side": MAX_SIDE,
            "models": {t: {k: (MODELS[t][k] is not None) for k in ORDER} for t in MODELS},
            "fusion_trained": FUSION_TRAINED}

@app.get("/benchmarks")
def benchmarks():
    return json.load(open(BENCH_FILE))

@app.post("/restore")
def restore(image: UploadFile = File(...), task: str = Form("deblur"), gt: Optional[UploadFile] = File(None)):
    if task not in MODELS: raise HTTPException(400, "task must be 'deblur'")
    try:
        img_bytes = image.file.read(); gt_bytes = gt.file.read() if gt is not None else None
        with gpu_lock:
            return run_pipeline(img_bytes, task, gt_bytes)
    except HTTPException: raise
    except Exception as e:
        raise HTTPException(500, f"{type(e).__name__}: {e}")

## 7. Start server and public tunnel
Copy the printed `https://....trycloudflare.com` URL into the website's "Backend URL" box.
Keep this notebook running while you demo. Quick tunnels close requests after about 100 s.

In [15]:
import uvicorn
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)

if not os.path.exists(f"{WORK}/cloudflared"):
    sh(f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {WORK}/cloudflared && chmod +x {WORK}/cloudflared")
tun = subprocess.Popen([f"{WORK}/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: public_url = m.group(0); break
print("\nBACKEND URL  ->", public_url)
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()   # keep draining the tunnel log

$ wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared

BACKEND URL  -> https://parenting-apr-associate-electronic.trycloudflare.com


## 8. Optional: smoke test from inside Colab

In [16]:
import requests
print(requests.get(f"http://localhost:{PORT}/health").json())

{'status': 'ok', 'gpu': 'Tesla T4', 'max_side': 1024, 'models': {'deblur': {'hinet': True, 'sfnet': True, 'adair': True}}, 'fusion_trained': {'deblur': False}}
